# 행동 분석 학습용 데이터 이상치 분석 (EDA)

이 노트북은 `behavior_preprocessed` 폴더의 학습용 CSV를 대상으로 다음을 점검합니다.

- 파일 단위 로딩 및 병합
- 컬럼 구조 / dtype / 기본 정보 확인
- 결측치 비율 확인
- 타깃 분포 및 클래스 불균형 확인
- 수치형 변수 기술통계 및 이상치 후보 탐색
- IQR / 분위수 기반 이상치 비율 확인
- 음수 / 비정상 범위 값 확인
- timestamp / sequence 정합성 확인
- sample 단위 길이 분포 및 이상 구간 확인
- train / valid 분포 차이 비교

> 기본 경로는 아래 코드 셀에서 수정할 수 있습니다.


In [1]:
# =========================================================
# 1. 라이브러리
# =========================================================
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


In [2]:
# =========================================================
# 2. 경로 설정
# =========================================================
PROJECT_ROOT = Path(r"C:\Users\codeit44\Desktop\side_project")
DATA_ROOT = PROJECT_ROOT / "output" / "behavior_preprocessed"

TRAIN_DIR = DATA_ROOT / "train_parts"
VALID_DIR = DATA_ROOT / "valid_parts"

SUMMARY_PATH = DATA_ROOT / "preprocess_summary.json"
FEATURE_INFO_PATH = DATA_ROOT / "feature_columns.json"
MAPPING_PATH = DATA_ROOT / "category_mappings.json"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)
print("TRAIN_DIR    :", TRAIN_DIR)
print("VALID_DIR    :", VALID_DIR)


PROJECT_ROOT : C:\Users\codeit44\Desktop\side_project
DATA_ROOT    : C:\Users\codeit44\Desktop\side_project\output\behavior_preprocessed
TRAIN_DIR    : C:\Users\codeit44\Desktop\side_project\output\behavior_preprocessed\train_parts
VALID_DIR    : C:\Users\codeit44\Desktop\side_project\output\behavior_preprocessed\valid_parts


In [3]:
# =========================================================
# 3. 파일 목록 확인
# =========================================================
train_files = sorted(TRAIN_DIR.glob("part_*.csv"))
valid_files = sorted(VALID_DIR.glob("part_*.csv"))

print(f"train files: {len(train_files)}")
print(f"valid files: {len(valid_files)}")

print("\n[train sample files]")
for f in train_files[:5]:
    print("-", f.name)

print("\n[valid sample files]")
for f in valid_files[:5]:
    print("-", f.name)

if SUMMARY_PATH.exists():
    with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
        preprocess_summary = json.load(f)
    print("\n[preprocess_summary]")
    print(json.dumps(preprocess_summary, ensure_ascii=False, indent=2))

if FEATURE_INFO_PATH.exists():
    with open(FEATURE_INFO_PATH, "r", encoding="utf-8") as f:
        feature_info = json.load(f)
    print("\n[feature_info]")
    print(json.dumps(feature_info, ensure_ascii=False, indent=2))


train files: 41
valid files: 6

[train sample files]
- part_0001.csv
- part_0002.csv
- part_0003.csv
- part_0004.csv
- part_0005.csv

[valid sample files]
- part_0000.csv
- part_0001.csv
- part_0002.csv
- part_0003.csv
- part_0004.csv

[preprocess_summary]
{
  "train_summary": {
    "split": "train",
    "num_input_parts": 42,
    "num_output_parts": 41,
    "input_rows": 43406022,
    "output_rows": 43305419,
    "b_rows_kept": 43305419,
    "target_non_null_rows": 43305419
  },
  "valid_summary": {
    "split": "valid",
    "num_input_parts": 6,
    "num_output_parts": 6,
    "input_rows": 5652859,
    "output_rows": 5643643,
    "b_rows_kept": 5643643,
    "target_non_null_rows": 5643643
  }
}

[feature_info]
{
  "target_col": "target_behavior",
  "target_id_col": "target_behavior_id",
  "feature_cols": [
    "em_temperature",
    "em_humidity",
    "em_illuminance",
    "em_activity_ir",
    "em_co2",
    "em_tvoc",
    "em_activity_ir_log1p",
    "em_co2_log1p",
    "em_tvoc_log1p

In [4]:
# =========================================================
# 4. 데이터 로드 함수
# =========================================================
USE_SAMPLE_ROWS = None   # 예: 2_000_000 으로 두면 일부만 로드
RANDOM_SEED = 42

def read_csv_safe(path, **kwargs):
    base_kwargs = {
        "encoding": "utf-8-sig",
        "low_memory": False
    }
    base_kwargs.update(kwargs)
    try:
        return pd.read_csv(path, **base_kwargs)
    except Exception:
        base_kwargs["encoding"] = "utf-8"
        return pd.read_csv(path, **base_kwargs)

def load_parts(file_list, split_name, use_sample_rows=None, random_state=42):
    dfs = []
    loaded_rows = 0

    for f in file_list:
        df = read_csv_safe(f)
        df["__source_file__"] = f.name
        df["__split__"] = split_name
        dfs.append(df)
        loaded_rows += len(df)
        print(f"[LOAD] {f.name} | rows={len(df):,}")

    if len(dfs) == 0:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)
    print(f"\n[{split_name}] total rows before sampling: {len(out):,}")

    if use_sample_rows is not None and len(out) > use_sample_rows:
        out = out.sample(use_sample_rows, random_state=random_state).sort_index().reset_index(drop=True)
        print(f"[{split_name}] sampled rows: {len(out):,}")

    return out


In [5]:
# =========================================================
# 5. train / valid 로드
# =========================================================
train_df = load_parts(train_files, "train", use_sample_rows=USE_SAMPLE_ROWS, random_state=RANDOM_SEED)
valid_df = load_parts(valid_files, "valid", use_sample_rows=USE_SAMPLE_ROWS, random_state=RANDOM_SEED)

print("\ntrain shape:", train_df.shape)
print("valid shape:", valid_df.shape)


[LOAD] part_0001.csv | rows=1,022,406
[LOAD] part_0002.csv | rows=1,313,068
[LOAD] part_0003.csv | rows=1,330,601
[LOAD] part_0004.csv | rows=1,327,829
[LOAD] part_0005.csv | rows=1,317,174
[LOAD] part_0006.csv | rows=1,329,432
[LOAD] part_0007.csv | rows=1,326,515
[LOAD] part_0008.csv | rows=1,352,309
[LOAD] part_0009.csv | rows=1,304,577
[LOAD] part_0010.csv | rows=1,278,641
[LOAD] part_0011.csv | rows=1,334,931
[LOAD] part_0012.csv | rows=1,240,797
[LOAD] part_0013.csv | rows=1,250,757
[LOAD] part_0014.csv | rows=1,331,656
[LOAD] part_0015.csv | rows=1,309,586
[LOAD] part_0016.csv | rows=1,367,716
[LOAD] part_0017.csv | rows=1,320,241
[LOAD] part_0018.csv | rows=1,331,830
[LOAD] part_0019.csv | rows=1,138,366
[LOAD] part_0020.csv | rows=1,092,601
[LOAD] part_0021.csv | rows=1,091,074
[LOAD] part_0022.csv | rows=1,070,733
[LOAD] part_0023.csv | rows=944,370
[LOAD] part_0024.csv | rows=1,233,310
[LOAD] part_0025.csv | rows=1,299,861
[LOAD] part_0026.csv | rows=1,322,174
[LOAD] part_00

In [6]:
# =========================================================
# 6. 기본 구조 확인
# =========================================================
def basic_overview(df, name):
    print(f"\n===== {name.upper()} OVERVIEW =====")
    print("shape:", df.shape)
    print("\ncolumns:")
    print(df.columns.tolist())
    print("\ndtypes:")
    print(df.dtypes)
    print("\nhead:")
    display(df.head())

basic_overview(train_df, "train")
basic_overview(valid_df, "valid")



===== TRAIN OVERVIEW =====
shape: (43305419, 25)

columns:
['sample_key', 'split', 'person_id', 'seq', 'seq_num', 'timestamp', 'target_behavior', 'target_behavior_id', 'em_temperature', 'em_humidity', 'em_illuminance', 'em_activity_ir', 'em_co2', 'em_tvoc', 'em_activity_ir_log1p', 'em_co2_log1p', 'em_tvoc_log1p', 'em_illuminance_log1p', 'hour', 'weekday', 'age', 'gender_id', 'environment_id', '__source_file__', '__split__']

dtypes:
sample_key               object
split                    object
person_id                object
seq                       int64
seq_num                   int64
timestamp                object
target_behavior          object
target_behavior_id        int64
em_temperature          float64
em_humidity             float64
em_illuminance          float64
em_activity_ir          float64
em_co2                  float64
em_tvoc                 float64
em_activity_ir_log1p    float64
em_co2_log1p            float64
em_tvoc_log1p           float64
em_illuminance_log

,sample_key,split,person_id,seq,seq_num,timestamp,target_behavior,target_behavior_id,em_temperature,em_humidity,em_illuminance,em_activity_ir,em_co2,em_tvoc,em_activity_ir_log1p,em_co2_log1p,em_tvoc_log1p,em_illuminance_log1p,hour,weekday,age,gender_id,environment_id,__source_file__,__split__
0,B0201|M|001,train,B0201,1,1000000,2023-06-08 17:40:00,기타,0,27.6,52.0,79.0,75.0,446.0,45.0,4.330733,6.102559,3.828641,4.382027,17.0,3.0,60.0,1,2,part_0001.csv,train
1,B0201|M|001,train,B0201,1,1000001,2023-06-23 13:00:00,기타,0,28.7,55.0,45.0,46.0,408.0,27.0,3.850148,6.013715,3.332205,3.828641,13.0,4.0,60.0,1,2,part_0001.csv,train
2,B0201|M|001,train,B0201,1,1000002,2023-06-23 13:10:00,기타,0,28.2,55.0,45.0,21.0,415.0,30.0,3.091042,6.030685,3.433987,3.828641,13.0,4.0,60.0,1,2,part_0001.csv,train
3,B0201|M|001,train,B0201,1,1000003,2023-06-23 13:20:00,기타,0,28.2,55.0,25.0,21.0,431.0,37.0,3.091042,6.068426,3.637586,3.258097,13.0,4.0,60.0,1,2,part_0001.csv,train
4,B0201|M|001,train,B0201,1,1000004,2023-06-23 13:30:00,기타,0,28.2,55.0,44.0,11.0,440.0,42.0,2.484907,6.089045,3.761200,3.806662,13.0,4.0,60.0,1,2,part_0001.csv,train



===== VALID OVERVIEW =====
shape: (5643643, 25)

columns:
['sample_key', 'split', 'person_id', 'seq', 'seq_num', 'timestamp', 'target_behavior', 'target_behavior_id', 'em_temperature', 'em_humidity', 'em_illuminance', 'em_activity_ir', 'em_co2', 'em_tvoc', 'em_activity_ir_log1p', 'em_co2_log1p', 'em_tvoc_log1p', 'em_illuminance_log1p', 'hour', 'weekday', 'age', 'gender_id', 'environment_id', '__source_file__', '__split__']

dtypes:
sample_key               object
split                    object
person_id                object
seq                       int64
seq_num                   int64
timestamp                object
target_behavior          object
target_behavior_id        int64
em_temperature          float64
em_humidity             float64
em_illuminance          float64
em_activity_ir          float64
em_co2                  float64
em_tvoc                 float64
em_activity_ir_log1p    float64
em_co2_log1p            float64
em_tvoc_log1p           float64
em_illuminance_log1

,sample_key,split,person_id,seq,seq_num,timestamp,target_behavior,target_behavior_id,em_temperature,em_humidity,em_illuminance,em_activity_ir,em_co2,em_tvoc,em_activity_ir_log1p,em_co2_log1p,em_tvoc_log1p,em_illuminance_log1p,hour,weekday,age,gender_id,environment_id,__source_file__,__split__
0,B0101|F|010,valid,B0101,10,1000000,2023-05-01 00:10:00,외출,3,21.1,47.0,35.0,0.0,407.0,0.0,0.0,6.011267,0.0,3.583519,0.0,0.0,60.0,0,2,part_0000.csv,valid
1,B0101|F|010,valid,B0101,10,1000001,2023-05-01 00:20:00,외출,3,21.1,47.0,35.0,0.0,403.0,0.0,0.0,6.001415,0.0,3.583519,0.0,0.0,60.0,0,2,part_0000.csv,valid
2,B0101|F|010,valid,B0101,10,1000002,2023-05-01 00:30:00,외출,3,21.1,47.0,33.0,0.0,402.0,0.0,0.0,5.998937,0.0,3.526361,0.0,0.0,60.0,0,2,part_0000.csv,valid
3,B0101|F|010,valid,B0101,10,1000003,2023-05-01 00:40:00,외출,3,21.1,47.0,34.0,0.0,402.0,0.0,0.0,5.998937,0.0,3.555348,0.0,0.0,60.0,0,2,part_0000.csv,valid
4,B0101|F|010,valid,B0101,10,1000004,2023-05-01 00:50:00,외출,3,21.1,47.0,35.0,0.0,402.0,0.0,0.0,5.998937,0.0,3.583519,0.0,0.0,60.0,0,2,part_0000.csv,valid


In [7]:
# =========================================================
# 7. 컬럼별 결측치 확인
# =========================================================
def missing_summary(df):
    ms = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_ratio": (df.isna().mean() * 100).round(4)
    }).sort_values(["missing_ratio", "missing_count"], ascending=False)
    return ms

train_missing = missing_summary(train_df)
valid_missing = missing_summary(valid_df)

print("[TRAIN missing]")
display(train_missing)

print("[VALID missing]")
display(valid_missing)


[TRAIN missing]


,missing_count,missing_ratio
sample_key,0,0.0
split,0,0.0
person_id,0,0.0
seq,0,0.0
seq_num,0,0.0
timestamp,0,0.0
target_behavior,0,0.0
target_behavior_id,0,0.0
em_temperature,0,0.0
em_humidity,0,0.0


[VALID missing]


,missing_count,missing_ratio
sample_key,0,0.0
split,0,0.0
person_id,0,0.0
seq,0,0.0
seq_num,0,0.0
timestamp,0,0.0
target_behavior,0,0.0
target_behavior_id,0,0.0
em_temperature,0,0.0
em_humidity,0,0.0


In [8]:
# =========================================================
# 8. 중복 확인
# =========================================================
key_cols = [c for c in ["sample_key", "seq_num", "timestamp"] if c in train_df.columns]

for name, df in [("train", train_df), ("valid", valid_df)]:
    print(f"\n===== {name.upper()} DUPLICATION CHECK =====")
    print("full row duplicates:", int(df.duplicated().sum()))
    if len(key_cols) > 0:
        print(f"key duplicates ({key_cols}):", int(df.duplicated(subset=key_cols).sum()))



===== TRAIN DUPLICATION CHECK =====
full row duplicates: 0
key duplicates (['sample_key', 'seq_num', 'timestamp']): 0

===== VALID DUPLICATION CHECK =====
full row duplicates: 0
key duplicates (['sample_key', 'seq_num', 'timestamp']): 0


In [9]:
# =========================================================
# 9. timestamp / seq 정합성 확인 - 메모리 절약 버전
# =========================================================

for df in [train_df, valid_df]:
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

def check_time_sequence_light(df, name):
    print(f"\n===== {name.upper()} TIME / SEQ CHECK =====")

    # 필요한 컬럼만 사용
    use_cols = [c for c in ["sample_key", "timestamp", "seq_num"] if c in df.columns]
    temp = df[use_cols].copy()

    if "timestamp" in temp.columns:
        print("timestamp null:", int(temp["timestamp"].isna().sum()))
        if temp["timestamp"].notna().any():
            print("timestamp min :", temp["timestamp"].min())
            print("timestamp max :", temp["timestamp"].max())

    # time diff 확인
    if {"sample_key", "timestamp"}.issubset(temp.columns):
        sort_cols = ["sample_key", "timestamp"]
        if "seq_num" in temp.columns:
            sort_cols.append("seq_num")

        time_temp = temp[["sample_key", "timestamp"] + (["seq_num"] if "seq_num" in temp.columns else [])]
        time_temp = time_temp.sort_values(sort_cols)

        time_diff_sec = time_temp.groupby("sample_key")["timestamp"].diff().dt.total_seconds()

        print("negative time diff rows:", int((time_diff_sec < 0).sum()))
        print("zero time diff rows    :", int((time_diff_sec == 0).sum()))
        print("large gap > 1 hour rows:", int((time_diff_sec > 3600).sum()))
        display(time_diff_sec.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

        # 긴 gap 샘플 일부 확인
        gap_mask = time_diff_sec > 3600
        if gap_mask.any():
            print("\n[large gap examples]")
            display(time_temp.loc[gap_mask, ["sample_key", "timestamp"]].head(20))

        del time_temp, time_diff_sec

    # seq diff 확인
    if {"sample_key", "seq_num"}.issubset(temp.columns):
        seq_temp = temp[["sample_key", "seq_num"]].copy()
        seq_temp = seq_temp.sort_values(["sample_key", "seq_num"])

        seq_diff = seq_temp.groupby("sample_key")["seq_num"].diff()

        print("\nnegative seq diff rows:", int((seq_diff < 0).sum()))
        print("zero seq diff rows    :", int((seq_diff == 0).sum()))
        print("seq jump > 1 rows     :", int((seq_diff > 1).sum()))
        display(seq_diff.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

        del seq_temp, seq_diff

    del temp

check_time_sequence_light(train_df, "train")
check_time_sequence_light(valid_df, "valid")


===== TRAIN TIME / SEQ CHECK =====
timestamp null: 0
timestamp min : 2023-05-01 00:10:00
timestamp max : 2024-08-31 23:50:00
negative time diff rows: 0
zero time diff rows    : 0
large gap > 1 hour rows: 7564


count    4.330462e+07
mean     6.311539e+02
std      1.184805e+04
min      6.000000e+02
50%      6.000000e+02
90%      6.000000e+02
95%      6.000000e+02
99%      6.000000e+02
max      3.079500e+07
Name: timestamp, dtype: float64


[large gap examples]


,sample_key,timestamp
1,B0201|M|001,2023-06-23 13:00:00
65119,B0202|M|001,2023-05-30 14:10:00
70429,B0202|M|001,2023-08-07 14:30:00
129071,B0203|F|001,2023-06-08 10:10:00
164099,B0203|F|001,2024-02-07 13:50:00
164397,B0203|F|001,2024-02-19 11:00:00
164403,B0203|F|001,2024-02-21 15:30:00
194529,B0204|F|001,2023-05-31 15:30:00
261175,B0205|F|001,2023-06-05 11:20:00
328859,B0206|F|001,2023-06-14 14:10:00



negative seq diff rows: 0
zero seq diff rows    : 0
seq jump > 1 rows     : 0


...

count    5.643543e+06
mean     6.807174e+02
std      3.811831e+04
min      6.000000e+02
50%      6.000000e+02
90%      6.000000e+02
95%      6.000000e+02
99%      6.000000e+02
max      3.564360e+07
Name: timestamp, dtype: float64


[large gap examples]


,sample_key,timestamp
74592,B0102|F|010,2023-06-18 12:30:00
74725,B0102|F|010,2023-06-23 15:40:00
74840,B0102|F|010,2023-07-08 14:50:00
74989,B0102|F|010,2023-07-11 19:40:00
74997,B0102|F|010,2023-07-12 18:00:00
75136,B0102|F|010,2023-07-21 07:50:00
75137,B0102|F|010,2023-09-07 10:10:00
81364,B0102|F|010,2023-10-23 15:50:00
100048,B0102|F|010,2024-03-11 10:10:00
100065,B0102|F|010,2024-03-13 10:00:00



negative seq diff rows: 0
zero seq diff rows    : 0
seq jump > 1 rows     : 0


count    5643543.0
mean           1.0
std            0.0
min            1.0
50%            1.0
90%            1.0
95%            1.0
99%            1.0
max            1.0
Name: seq_num, dtype: float64

In [11]:
# =========================================================
# 10. sample 길이 분포 확인 - 그래프 없는 버전
# =========================================================
def sample_length_summary(df, name):
    if "sample_key" not in df.columns:
        print(f"{name}: sample_key 없음")
        return None

    lengths = df.groupby("sample_key").size().rename("num_rows").reset_index()

    print(f"\n===== {name.upper()} SAMPLE LENGTH =====")
    display(lengths["num_rows"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

    print("\n구간별 개수")
    bins = [0, 10000, 30000, 50000, 60000, 65000, 70000, 70500, 100000]
    labels = [
        "0~1만",
        "1만~3만",
        "3만~5만",
        "5만~6만",
        "6만~6.5만",
        "6.5만~7만",
        "7만~7.05만",
        "7.05만 이상",
    ]

    lengths["length_bin"] = pd.cut(
        lengths["num_rows"],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    display(
        lengths["length_bin"]
        .value_counts()
        .sort_index()
        .rename("sample_count")
        .reset_index()
    )

    print("\nTop 20 longest samples")
    display(lengths.sort_values("num_rows", ascending=False).head(20))

    print("\nTop 20 shortest samples")
    display(lengths.sort_values("num_rows", ascending=True).head(20))

    return lengths

train_lengths = sample_length_summary(train_df, "train")
valid_lengths = sample_length_summary(valid_df, "valid")


===== TRAIN SAMPLE LENGTH =====


count      800.000000
mean     54131.773750
std      16898.621901
min       5705.000000
50%      61707.500000
90%      70003.400000
95%      70384.250000
99%      70413.010000
max      70415.000000
Name: num_rows, dtype: float64


구간별 개수


,length_bin,sample_count
0,0~1만,6
1,1만~3만,75
2,3만~5만,206
3,5만~6만,88
4,6만~6.5만,83
5,6.5만~7만,262
6,7만~7.05만,80
7,7.05만 이상,0



Top 20 longest samples


...

,length_bin,sample_count
0,0~1만,4
1,1만~3만,14
2,3만~5만,3
3,5만~6만,10
4,6만~6.5만,12
5,6.5만~7만,36
6,7만~7.05만,21
7,7.05만 이상,0



Top 20 longest samples


,sample_key,num_rows,length_bin
66,B0167|F|001,70415,7만~7.05만
67,B0168|F|001,70415,7만~7.05만
74,B0175|F|001,70415,7만~7.05만
88,B0189|F|001,70413,7만~7.05만
75,B0176|M|001,70411,7만~7.05만
61,B0162|F|001,70409,7만~7.05만
62,B0163|F|001,70401,7만~7.05만
94,B0195|F|001,70400,7만~7.05만
77,B0178|F|001,70400,7만~7.05만
72,B0173|M|001,70396,7만~7.05만



Top 20 shortest samples


,sample_key,num_rows,length_bin
30,B0131|F|001,2031,0~1만
25,B0126|F|001,6599,0~1만
16,B0117|F|001,9292,0~1만
57,B0158|F|001,9398,0~1만
24,B0125|F|001,10430,1만~3만
8,B0109|F|001,10565,1만~3만
15,B0116|F|001,10567,1만~3만
44,B0145|F|001,10712,1만~3만
39,B0140|F|001,10714,1만~3만
56,B0157|F|001,10853,1만~3만


In [13]:
# =========================================================
# 11. 타깃 분포 확인 - 그래프 없는 버전
# =========================================================
TARGET_COL = "target_behavior"
TARGET_ID_COL = "target_behavior_id"

def target_distribution(df, name):
    print(f"\n===== {name.upper()} TARGET DISTRIBUTION =====")

    if TARGET_COL in df.columns:
        vc = df[TARGET_COL].value_counts(dropna=False)
        ratio = (df[TARGET_COL].value_counts(normalize=True, dropna=False) * 100).round(4)

        out = pd.DataFrame({
            "target_behavior": vc.index,
            "count": vc.values,
            "ratio_%": ratio.values,
        })

        display(out)

        print("\n[Text bar]")
        max_count = out["count"].max()
        for _, row in out.iterrows():
            bar_len = int(row["count"] / max_count * 40)
            bar = "█" * bar_len
            print(f"{row['target_behavior']:<10} | {bar} {row['count']:,} ({row['ratio_%']}%)")

        imbalance_ratio = vc.max() / max(vc.min(), 1)
        print("\nimbalance ratio (max/min):", round(float(imbalance_ratio), 4))

    if TARGET_ID_COL in df.columns:
        print(f"\n{name} target_id unique count:", df[TARGET_ID_COL].nunique(dropna=True))
        print(f"{name} target_id == -1 rows:", int((df[TARGET_ID_COL] == -1).sum()))

target_distribution(train_df, "train")
target_distribution(valid_df, "valid")


===== TRAIN TARGET DISTRIBUTION =====


,target_behavior,count,ratio_%
0,기타,18712091,43.2096
1,수면,15497731,35.7870
2,외출,8513153,19.6584
3,식사,582444,1.3450



[Text bar]
기타         | ████████████████████████████████████████ 18,712,091 (43.2096%)
수면         | █████████████████████████████████ 15,497,731 (35.787%)
외출         | ██████████████████ 8,513,153 (19.6584%)
식사         | █ 582,444 (1.345%)

imbalance ratio (max/min): 32.1268

train target_id unique count: 4
train target_id == -1 rows: 0

===== VALID TARGET DISTRIBUTION =====


,target_behavior,count,ratio_%
0,기타,2375106,42.0846
1,수면,2112760,37.4361
2,외출,1093567,19.3770
3,식사,62210,1.1023



[Text bar]
기타         | ████████████████████████████████████████ 2,375,106 (42.0846%)
수면         | ███████████████████████████████████ 2,112,760 (37.4361%)
외출         | ██████████████████ 1,093,567 (19.377%)
식사         | █ 62,210 (1.1023%)

imbalance ratio (max/min): 38.1788

valid target_id unique count: 4
valid target_id == -1 rows: 0


In [14]:
# =========================================================
# 12. 수치형 컬럼 자동 추출
# =========================================================
exclude_cols = {
    "sample_key", "split", "__split__", "__source_file__", "person_id", "seq", "timestamp",
    "target_behavior"
}

numeric_cols = []
for c in train_df.columns:
    if c in exclude_cols:
        continue
    if pd.api.types.is_numeric_dtype(train_df[c]):
        numeric_cols.append(c)

print("numeric_cols:")
print(numeric_cols)


numeric_cols:
['seq_num', 'target_behavior_id', 'em_temperature', 'em_humidity', 'em_illuminance', 'em_activity_ir', 'em_co2', 'em_tvoc', 'em_activity_ir_log1p', 'em_co2_log1p', 'em_tvoc_log1p', 'em_illuminance_log1p', 'hour', 'weekday', 'age', 'gender_id', 'environment_id']


In [15]:
# =========================================================
# 13. 수치형 기술통계
# =========================================================
def numeric_summary(df, cols, name):
    if len(cols) == 0:
        print(f"{name}: numeric columns 없음")
        return
    desc = df[cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc["missing_ratio_%"] = (df[cols].isna().mean() * 100).round(4)
    print(f"\n===== {name.upper()} NUMERIC SUMMARY =====")
    display(desc)

numeric_summary(train_df, numeric_cols, "train")
numeric_summary(valid_df, numeric_cols, "valid")



===== TRAIN NUMERIC SUMMARY =====


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_ratio_%
seq_num,43305419.0,1.029700e+06,18794.860027,1000000.000000,1.000541e+06,1.002706e+06,1.013598e+06,1.027992e+06,1.044524e+06,1.062422e+06,1.067613e+06,1.070414e+06,0.0
target_behavior_id,43305419.0,9.745219e-01,1.109609,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,0.0
em_temperature,43305419.0,2.603736e+01,4.209846,-0.600000,1.340000e+01,1.820000e+01,2.360000e+01,2.650000e+01,2.910000e+01,3.180000e+01,3.330000e+01,5.320000e+01,0.0
em_humidity,43305419.0,5.319055e+01,14.727853,0.000000,2.100000e+01,2.800000e+01,4.300000e+01,5.300000e+01,6.400000e+01,7.600000e+01,8.300000e+01,9.500000e+01,0.0
em_illuminance,43305419.0,1.642821e+01,19.801350,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,6.000000e+00,3.200000e+01,5.500000e+01,6.700000e+01,9.900000e+01,0.0
em_activity_ir,43305419.0,1.967181e+01,46.882803,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.700000e+01,1.060000e+02,2.370000e+02,5.870000e+02,0.0
em_co2,43305419.0,4.315742e+02,55.356588,400.000000,4.000000e+02,4.010000e+02,4.120000e+02,4.160000e+02,4.290000e+02,5.130000e+02,6.630000e+02,2.871900e+04,0.0
em_tvoc,43305419.0,3.016862e+01,62.908946,0.000000,0.000000e+00,0.000000e+00,2.200000e+01,2.900000e+01,3.200000e+01,5.900000e+01,1.470000e+02,3.426000e+04,0.0
em_activity_ir_log1p,43305419.0,1.338255e+00,1.771024,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.890372e+00,4.672829e+00,5.472271e+00,6.376727e+00,0.0
em_co2_log1p,43305419.0,6.064538e+00,0.094926,5.993961,5.993961e+00,5.996452e+00,6.023448e+00,6.033086e+00,6.063785e+00,6.242223e+00,6.498282e+00,1.026535e+01,0.0



===== VALID NUMERIC SUMMARY =====


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing_ratio_%
seq_num,5643643.0,1.032024e+06,19708.807977,1000000.000000,1.000564e+06,1.002829e+06,1.014703e+06,1.031133e+06,1.048633e+06,1.064243e+06,1.068529e+06,1.070414e+06,0.0
target_behavior_id,5643643.0,9.777162e-01,1.098386,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,3.000000e+00,3.000000e+00,0.0
em_temperature,5643643.0,2.640796e+01,3.848067,3.700000,1.520000e+01,1.940000e+01,2.420000e+01,2.680000e+01,2.930000e+01,3.170000e+01,3.310000e+01,3.840000e+01,0.0
em_humidity,5643643.0,5.375856e+01,14.409033,5.000000,1.900000e+01,2.800000e+01,4.400000e+01,5.500000e+01,6.500000e+01,7.600000e+01,8.200000e+01,9.500000e+01,0.0
em_illuminance,5643643.0,1.268822e+01,17.321392,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,2.300000e+01,4.900000e+01,6.300000e+01,9.900000e+01,0.0
em_activity_ir,5643643.0,1.915990e+01,46.082102,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.700000e+01,1.000000e+02,2.340000e+02,5.720000e+02,0.0
em_co2,5643643.0,4.317454e+02,53.784685,400.000000,4.000000e+02,4.010000e+02,4.120000e+02,4.180000e+02,4.330000e+02,4.990000e+02,6.320000e+02,1.033100e+04,0.0
em_tvoc,5643643.0,2.652119e+01,99.749419,0.000000,0.000000e+00,0.000000e+00,4.000000e+00,2.900000e+01,3.200000e+01,5.000000e+01,1.220000e+02,1.857400e+04,0.0
em_activity_ir_log1p,5643643.0,1.345564e+00,1.757246,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.890372e+00,4.615121e+00,5.459586e+00,6.350886e+00,0.0
em_co2_log1p,5643643.0,6.065213e+00,0.091192,5.993961,5.993961e+00,5.996452e+00,6.023448e+00,6.037871e+00,6.073045e+00,6.214608e+00,6.450470e+00,9.243001e+00,0.0


In [16]:
# =========================================================
# 14. 범위 이상치 / 논리 이상치 확인
# =========================================================
def rule_based_checks(df, name):
    print(f"\n===== {name.upper()} RULE-BASED CHECKS =====")

    rules = {}
    if "age" in df.columns:
        rules["age < 0 or > 120"] = ((df["age"] < 0) | (df["age"] > 120)).fillna(False)
    if "hour" in df.columns:
        rules["hour < 0 or > 23"] = ((df["hour"] < 0) | (df["hour"] > 23)).fillna(False)
    if "weekday" in df.columns:
        rules["weekday < 0 or > 6"] = ((df["weekday"] < 0) | (df["weekday"] > 6)).fillna(False)
    if "em_humidity" in df.columns:
        rules["humidity < 0 or > 100"] = ((df["em_humidity"] < 0) | (df["em_humidity"] > 100)).fillna(False)
    for c in ["em_temperature", "em_illuminance", "em_activity_ir", "em_co2", "em_tvoc",
              "em_activity_ir_log1p", "em_co2_log1p", "em_tvoc_log1p", "em_illuminance_log1p"]:
        if c in df.columns:
            rules[f"{c} < 0"] = (df[c] < 0).fillna(False)

    result = []
    for rule_name, mask in rules.items():
        result.append({
            "rule": rule_name,
            "count": int(mask.sum()),
            "ratio_%": round(mask.mean() * 100, 6)
        })

    result = pd.DataFrame(result).sort_values(["count", "ratio_%"], ascending=False)
    display(result)

rule_based_checks(train_df, "train")
rule_based_checks(valid_df, "valid")



===== TRAIN RULE-BASED CHECKS =====


,rule,count,ratio_%
4,em_temperature < 0,31,0.000072
0,age < 0 or > 120,0,0.000000
1,hour < 0 or > 23,0,0.000000
2,weekday < 0 or > 6,0,0.000000
3,humidity < 0 or > 100,0,0.000000
5,em_illuminance < 0,0,0.000000
6,em_activity_ir < 0,0,0.000000
7,em_co2 < 0,0,0.000000
8,em_tvoc < 0,0,0.000000
9,em_activity_ir_log1p < 0,0,0.000000



===== VALID RULE-BASED CHECKS =====


,rule,count,ratio_%
0,age < 0 or > 120,0,0.0
1,hour < 0 or > 23,0,0.0
2,weekday < 0 or > 6,0,0.0
3,humidity < 0 or > 100,0,0.0
4,em_temperature < 0,0,0.0
5,em_illuminance < 0,0,0.0
6,em_activity_ir < 0,0,0.0
7,em_co2 < 0,0,0.0
8,em_tvoc < 0,0,0.0
9,em_activity_ir_log1p < 0,0,0.0


In [17]:
# =========================================================
# 15. IQR 기반 이상치 비율 확인
# =========================================================
def iqr_outlier_summary(df, cols):
    rows = []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(s) == 0:
            continue

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_mask = (pd.to_numeric(df[c], errors="coerce") < lower) | (pd.to_numeric(df[c], errors="coerce") > upper)

        rows.append({
            "column": c,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(outlier_mask.fillna(False).sum()),
            "outlier_ratio_%": round(outlier_mask.fillna(False).mean() * 100, 4)
        })
    return pd.DataFrame(rows).sort_values(["outlier_ratio_%", "outlier_count"], ascending=False)

train_iqr = iqr_outlier_summary(train_df, numeric_cols)
valid_iqr = iqr_outlier_summary(valid_df, numeric_cols)

print("[TRAIN IQR outlier summary]")
display(train_iqr)

print("[VALID IQR outlier summary]")
display(valid_iqr)


[TRAIN IQR outlier summary]


,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_ratio_%
10,em_tvoc_log1p,3.135494e+00,3.496508e+00,0.361013,2.593974,4.038028e+00,12400681,28.6354
7,em_tvoc,2.200000e+01,3.200000e+01,10.000000,7.000000,4.700000e+01,11818492,27.2910
1,target_behavior_id,0.000000e+00,1.000000e+00,1.000000,-1.500000,2.500000e+00,8513153,19.6584
5,em_activity_ir,0.000000e+00,1.700000e+01,17.000000,-25.500000,4.250000e+01,5935656,13.7065
6,em_co2,4.120000e+02,4.290000e+02,17.000000,386.500000,4.545000e+02,5649882,13.0466
9,em_co2_log1p,6.023448e+00,6.063785e+00,0.040338,5.962941,6.124292e+00,5533662,12.7782
14,age,6.000000e+01,6.000000e+01,0.000000,60.000000,6.000000e+01,2239630,5.1717
2,em_temperature,2.360000e+01,2.910000e+01,5.500000,15.350000,3.735000e+01,839204,1.9379
4,em_illuminance,0.000000e+00,3.200000e+01,32.000000,-48.000000,8.000000e+01,43075,0.0995
3,em_humidity,4.300000e+01,6.400000e+01,21.000000,11.500000,9.550000e+01,13641,0.0315


[VALID IQR outlier summary]


,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_ratio_%
1,target_behavior_id,0.000000e+00,1.000000e+00,1.000000,-1.500000,2.500000e+00,1093567,19.3770
15,gender_id,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,1044004,18.4988
5,em_activity_ir,0.000000e+00,1.700000e+01,17.000000,-25.500000,4.250000e+01,744913,13.1992
6,em_co2,4.120000e+02,4.330000e+02,21.000000,380.500000,4.645000e+02,613161,10.8646
9,em_co2_log1p,6.023448e+00,6.073045e+00,0.049597,5.949052,6.147440e+00,582959,10.3295
7,em_tvoc,4.000000e+00,3.200000e+01,28.000000,-38.000000,7.400000e+01,129925,2.3021
4,em_illuminance,0.000000e+00,2.300000e+01,23.000000,-34.500000,5.750000e+01,94374,1.6722
2,em_temperature,2.420000e+01,2.930000e+01,5.100000,16.550000,3.695000e+01,88228,1.5633
10,em_tvoc_log1p,1.609438e+00,3.496508e+00,1.887070,-1.221167,6.327112e+00,7939,0.1407
3,em_humidity,4.400000e+01,6.500000e+01,21.000000,12.500000,9.650000e+01,7562,0.1340


In [18]:
# =========================================================
# 16. 분위수 기반 극단치 확인
# =========================================================
def quantile_extreme_summary(df, cols, low_q=0.001, high_q=0.999):
    rows = []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        if len(s) == 0:
            continue

        low = s.quantile(low_q)
        high = s.quantile(high_q)
        mask = (pd.to_numeric(df[c], errors="coerce") < low) | (pd.to_numeric(df[c], errors="coerce") > high)

        rows.append({
            "column": c,
            f"q{low_q}": low,
            f"q{high_q}": high,
            "extreme_count": int(mask.fillna(False).sum()),
            "extreme_ratio_%": round(mask.fillna(False).mean() * 100, 4)
        })
    return pd.DataFrame(rows).sort_values(["extreme_ratio_%", "extreme_count"], ascending=False)

train_extreme = quantile_extreme_summary(train_df, numeric_cols)
valid_extreme = quantile_extreme_summary(valid_df, numeric_cols)

print("[TRAIN quantile extreme summary]")
display(train_extreme)

print("[VALID quantile extreme summary]")
display(valid_extreme)


[TRAIN quantile extreme summary]


,column,q0.001,q0.999,extreme_count,extreme_ratio_%
0,seq_num,1.000054e+06,1.069820e+06,86506,0.1998
2,em_temperature,8.100000e+00,3.510000e+01,81022,0.1871
3,em_humidity,1.400000e+01,9.100000e+01,71853,0.1659
7,em_tvoc,0.000000e+00,5.640000e+02,43237,0.0998
10,em_tvoc_log1p,0.000000e+00,6.336826e+00,43237,0.0998
5,em_activity_ir,0.000000e+00,4.270000e+02,43189,0.0997
8,em_activity_ir_log1p,0.000000e+00,6.059123e+00,43189,0.0997
6,em_co2,4.000000e+02,8.930000e+02,43091,0.0995
9,em_co2_log1p,5.993961e+00,6.795706e+00,43091,0.0995
4,em_illuminance,0.000000e+00,8.000000e+01,43075,0.0995


[VALID quantile extreme summary]


,column,q0.001,q0.999,extreme_count,extreme_ratio_%
0,seq_num,1.000056e+06,1.070055e+06,11235,0.1991
2,em_temperature,9.200000e+00,3.450000e+01,10739,0.1903
3,em_humidity,1.200000e+01,8.700000e+01,9418,0.1669
7,em_tvoc,0.000000e+00,7.750000e+02,5642,0.1000
10,em_tvoc_log1p,0.000000e+00,6.654153e+00,5642,0.1000
6,em_co2,4.000000e+02,9.870000e+02,5639,0.0999
9,em_co2_log1p,5.993961e+00,6.895683e+00,5639,0.0999
5,em_activity_ir,0.000000e+00,4.350000e+02,5616,0.0995
8,em_activity_ir_log1p,0.000000e+00,6.077642e+00,5616,0.0995
4,em_illuminance,0.000000e+00,8.000000e+01,4759,0.0843


In [20]:
# =========================================================
# 18. 범주형 / ID 컬럼 점검
# =========================================================
cat_like_cols = [c for c in ["gender_id", "environment_id", "target_behavior_id"] if c in train_df.columns]

for c in cat_like_cols:
    print(f"\n===== {c} =====")
    print("[TRAIN]")
    display(train_df[c].value_counts(dropna=False).sort_index())
    print("[VALID]")
    display(valid_df[c].value_counts(dropna=False).sort_index())



===== gender_id =====
[TRAIN]


gender_id
0    31499700
1    11805719
Name: count, dtype: int64

[VALID]


gender_id
0    4599639
1    1044004
Name: count, dtype: int64


===== environment_id =====
[TRAIN]


...

environment_id
0     561438
1    1102063
2    3980142
Name: count, dtype: int64


===== target_behavior_id =====
[TRAIN]


target_behavior_id
0    18712091
1    15497731
2      582444
3     8513153
Name: count, dtype: int64

[VALID]


target_behavior_id
0    2375106
1    2112760
2      62210
3    1093567
Name: count, dtype: int64

In [21]:
# =========================================================
# 19. train / valid 분포 차이 비교
# =========================================================
def compare_numeric_distribution(train_df, valid_df, cols):
    rows = []
    for c in cols:
        t = pd.to_numeric(train_df[c], errors="coerce")
        v = pd.to_numeric(valid_df[c], errors="coerce")
        rows.append({
            "column": c,
            "train_mean": t.mean(),
            "valid_mean": v.mean(),
            "mean_diff": v.mean() - t.mean(),
            "train_std": t.std(),
            "valid_std": v.std(),
            "train_q50": t.quantile(0.5),
            "valid_q50": v.quantile(0.5),
            "train_q95": t.quantile(0.95),
            "valid_q95": v.quantile(0.95),
        })
    return pd.DataFrame(rows).sort_values("mean_diff", key=lambda s: s.abs(), ascending=False)

dist_compare = compare_numeric_distribution(train_df, valid_df, numeric_cols)
display(dist_compare)


,column,train_mean,valid_mean,mean_diff,train_std,valid_std,train_q50,valid_q50,train_q95,valid_q95
0,seq_num,1.029700e+06,1.032024e+06,2323.748363,18794.860027,19708.807977,1.027992e+06,1.031133e+06,1.062422e+06,1.064243e+06
4,em_illuminance,1.642821e+01,1.268822e+01,-3.739991,19.801350,17.321392,6.000000e+00,2.000000e+00,5.500000e+01,4.900000e+01
7,em_tvoc,3.016862e+01,2.652119e+01,-3.647434,62.908946,99.749419,2.900000e+01,2.900000e+01,5.900000e+01,5.000000e+01
14,age,5.939264e+01,6.000000e+01,0.607360,2.753313,0.000000,6.000000e+01,6.000000e+01,6.000000e+01,6.000000e+01
3,em_humidity,5.319055e+01,5.375856e+01,0.568016,14.727853,14.409033,5.300000e+01,5.500000e+01,7.600000e+01,7.600000e+01
5,em_activity_ir,1.967181e+01,1.915990e+01,-0.511908,46.882803,46.082102,0.000000e+00,0.000000e+00,1.060000e+02,1.000000e+02
2,em_temperature,2.603736e+01,2.640796e+01,0.370608,4.209846,3.848067,2.650000e+01,2.680000e+01,3.180000e+01,3.170000e+01
10,em_tvoc_log1p,2.924583e+00,2.610369e+00,-0.314214,1.261318,1.367183,3.401197e+00,3.401197e+00,4.094345e+00,3.931826e+00
11,em_illuminance_log1p,1.782058e+00,1.520145e+00,-0.261913,1.668716,1.594949,1.945910e+00,1.098612e+00,4.025352e+00,3.912023e+00
16,environment_id,1.418297e+00,1.605762e+00,0.187464,0.703071,0.661648,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00


In [22]:
# =========================================================
# 20. 이상치 후보 샘플 추출
# =========================================================
def extract_outlier_examples(df, iqr_df, top_n_cols=5, top_k_rows=20):
    if len(iqr_df) == 0:
        return {}

    results = {}
    for _, row in iqr_df.head(top_n_cols).iterrows():
        c = row["column"]
        lower = row["lower_bound"]
        upper = row["upper_bound"]
        mask = (pd.to_numeric(df[c], errors="coerce") < lower) | (pd.to_numeric(df[c], errors="coerce") > upper)
        cols = [x for x in ["sample_key", "seq_num", "timestamp", "target_behavior", c, "__source_file__"] if x in df.columns]
        results[c] = df.loc[mask.fillna(False), cols].head(top_k_rows).copy()
    return results

train_outlier_examples = extract_outlier_examples(train_df, train_iqr, top_n_cols=5, top_k_rows=20)

for col_name, ex_df in train_outlier_examples.items():
    print(f"\n===== TRAIN OUTLIER EXAMPLES: {col_name} =====")
    display(ex_df)



===== TRAIN OUTLIER EXAMPLES: em_tvoc_log1p =====


,sample_key,seq_num,timestamp,target_behavior,em_tvoc_log1p,__source_file__
31,B0201|M|001,1000031,2023-06-23 18:00:00,식사,4.290459,part_0001.csv
47,B0201|M|001,1000047,2023-06-23 20:40:00,식사,4.369448,part_0001.csv
56,B0201|M|001,1000056,2023-06-23 22:10:00,기타,4.343805,part_0001.csv
65,B0201|M|001,1000065,2023-06-23 23:40:00,수면,4.158883,part_0001.csv
66,B0201|M|001,1000066,2023-06-23 23:50:00,수면,4.060443,part_0001.csv
92,B0201|M|001,1000092,2023-06-24 04:10:00,식사,4.488636,part_0001.csv
120,B0201|M|001,1000120,2023-06-24 08:50:00,식사,4.624973,part_0001.csv
154,B0201|M|001,1000154,2023-06-24 14:30:00,식사,4.779123,part_0001.csv
192,B0201|M|001,1000192,2023-06-24 20:50:00,식사,4.564348,part_0001.csv
212,B0201|M|001,1000212,2023-06-25 00:10:00,수면,4.890349,part_0001.csv



===== TRAIN OUTLIER EXAMPLES: em_tvoc =====


,sample_key,seq_num,timestamp,target_behavior,em_tvoc,__source_file__
5,B0201|M|001,1000005,2023-06-23 13:40:00,기타,53.0,part_0001.csv
31,B0201|M|001,1000031,2023-06-23 18:00:00,식사,72.0,part_0001.csv
47,B0201|M|001,1000047,2023-06-23 20:40:00,식사,78.0,part_0001.csv
56,B0201|M|001,1000056,2023-06-23 22:10:00,기타,76.0,part_0001.csv
65,B0201|M|001,1000065,2023-06-23 23:40:00,수면,63.0,part_0001.csv
66,B0201|M|001,1000066,2023-06-23 23:50:00,수면,57.0,part_0001.csv
92,B0201|M|001,1000092,2023-06-24 04:10:00,식사,88.0,part_0001.csv
110,B0201|M|001,1000110,2023-06-24 07:10:00,기타,52.0,part_0001.csv
118,B0201|M|001,1000118,2023-06-24 08:30:00,기타,48.0,part_0001.csv
120,B0201|M|001,1000120,2023-06-24 08:50:00,식사,101.0,part_0001.csv



===== TRAIN OUTLIER EXAMPLES: target_behavior_id =====


,sample_key,seq_num,timestamp,target_behavior,target_behavior_id,__source_file__
126,B0201|M|001,1000126,2023-06-24 09:50:00,외출,3,part_0001.csv
127,B0201|M|001,1000127,2023-06-24 10:00:00,외출,3,part_0001.csv
128,B0201|M|001,1000128,2023-06-24 10:10:00,외출,3,part_0001.csv
129,B0201|M|001,1000129,2023-06-24 10:20:00,외출,3,part_0001.csv
130,B0201|M|001,1000130,2023-06-24 10:30:00,외출,3,part_0001.csv
131,B0201|M|001,1000131,2023-06-24 10:40:00,외출,3,part_0001.csv
132,B0201|M|001,1000132,2023-06-24 10:50:00,외출,3,part_0001.csv
133,B0201|M|001,1000133,2023-06-24 11:00:00,외출,3,part_0001.csv
134,B0201|M|001,1000134,2023-06-24 11:10:00,외출,3,part_0001.csv
135,B0201|M|001,1000135,2023-06-24 11:20:00,외출,3,part_0001.csv



===== TRAIN OUTLIER EXAMPLES: em_activity_ir =====


,sample_key,seq_num,timestamp,target_behavior,em_activity_ir,__source_file__
0,B0201|M|001,1000000,2023-06-08 17:40:00,기타,75.0,part_0001.csv
1,B0201|M|001,1000001,2023-06-23 13:00:00,기타,46.0,part_0001.csv
7,B0201|M|001,1000007,2023-06-23 14:00:00,기타,52.0,part_0001.csv
8,B0201|M|001,1000008,2023-06-23 14:10:00,기타,45.0,part_0001.csv
20,B0201|M|001,1000020,2023-06-23 16:10:00,기타,47.0,part_0001.csv
26,B0201|M|001,1000026,2023-06-23 17:10:00,기타,46.0,part_0001.csv
31,B0201|M|001,1000031,2023-06-23 18:00:00,식사,69.0,part_0001.csv
35,B0201|M|001,1000035,2023-06-23 18:40:00,기타,58.0,part_0001.csv
36,B0201|M|001,1000036,2023-06-23 18:50:00,기타,88.0,part_0001.csv
53,B0201|M|001,1000053,2023-06-23 21:40:00,기타,44.0,part_0001.csv



===== TRAIN OUTLIER EXAMPLES: em_co2 =====


,sample_key,seq_num,timestamp,target_behavior,em_co2,__source_file__
5,B0201|M|001,1000005,2023-06-23 13:40:00,기타,455.0,part_0001.csv
31,B0201|M|001,1000031,2023-06-23 18:00:00,식사,488.0,part_0001.csv
47,B0201|M|001,1000047,2023-06-23 20:40:00,식사,503.0,part_0001.csv
56,B0201|M|001,1000056,2023-06-23 22:10:00,기타,505.0,part_0001.csv
65,B0201|M|001,1000065,2023-06-23 23:40:00,수면,474.0,part_0001.csv
66,B0201|M|001,1000066,2023-06-23 23:50:00,수면,471.0,part_0001.csv
92,B0201|M|001,1000092,2023-06-24 04:10:00,식사,519.0,part_0001.csv
110,B0201|M|001,1000110,2023-06-24 07:10:00,기타,460.0,part_0001.csv
120,B0201|M|001,1000120,2023-06-24 08:50:00,식사,511.0,part_0001.csv
154,B0201|M|001,1000154,2023-06-24 14:30:00,식사,559.0,part_0001.csv


In [23]:
# =========================================================
# 21. 전처리 방향 결론용 체크리스트
# =========================================================
checklist = {
    "target_null_exists": bool(train_df["target_behavior"].isna().sum() > 0) if "target_behavior" in train_df.columns else None,
    "target_id_minus1_exists_train": bool((train_df["target_behavior_id"] == -1).sum() > 0) if "target_behavior_id" in train_df.columns else None,
    "target_id_minus1_exists_valid": bool((valid_df["target_behavior_id"] == -1).sum() > 0) if "target_behavior_id" in valid_df.columns else None,
    "timestamp_null_exists_train": bool(train_df["timestamp"].isna().sum() > 0) if "timestamp" in train_df.columns else None,
    "full_row_duplicates_train": int(train_df.duplicated().sum()),
    "full_row_duplicates_valid": int(valid_df.duplicated().sum()),
}

print(json.dumps(checklist, ensure_ascii=False, indent=2))


{
  "target_null_exists": false,
  "target_id_minus1_exists_train": false,
  "target_id_minus1_exists_valid": false,
  "timestamp_null_exists_train": false,
  "full_row_duplicates_train": 0,
  "full_row_duplicates_valid": 0
}


In [24]:
# =========================================================
# 22. EDA 요약 저장
# =========================================================
SAVE_ANALYSIS_DIR = DATA_ROOT / "eda_results"
SAVE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

train_missing.to_csv(SAVE_ANALYSIS_DIR / "train_missing_summary.csv", encoding="utf-8-sig")
valid_missing.to_csv(SAVE_ANALYSIS_DIR / "valid_missing_summary.csv", encoding="utf-8-sig")

if len(train_iqr) > 0:
    train_iqr.to_csv(SAVE_ANALYSIS_DIR / "train_iqr_outlier_summary.csv", index=False, encoding="utf-8-sig")
if len(valid_iqr) > 0:
    valid_iqr.to_csv(SAVE_ANALYSIS_DIR / "valid_iqr_outlier_summary.csv", index=False, encoding="utf-8-sig")

if len(train_extreme) > 0:
    train_extreme.to_csv(SAVE_ANALYSIS_DIR / "train_quantile_extreme_summary.csv", index=False, encoding="utf-8-sig")
if len(valid_extreme) > 0:
    valid_extreme.to_csv(SAVE_ANALYSIS_DIR / "valid_quantile_extreme_summary.csv", index=False, encoding="utf-8-sig")

dist_compare.to_csv(SAVE_ANALYSIS_DIR / "train_valid_distribution_compare.csv", index=False, encoding="utf-8-sig")

with open(SAVE_ANALYSIS_DIR / "eda_checklist.json", "w", encoding="utf-8") as f:
    json.dump(checklist, f, ensure_ascii=False, indent=2)

print("saved to:", SAVE_ANALYSIS_DIR)


saved to: C:\Users\codeit44\Desktop\side_project\output\behavior_preprocessed\eda_results


## 해석 가이드

아래 항목을 보고 다음 전처리 방향을 결정하면 됩니다.

1. `missing_ratio`가 높은 컬럼  
   - 삭제할지
   - 대체값으로 채울지
   - 모델 입력에서 제외할지 결정

2. `target_behavior_id == -1` 존재 여부  
   - mapping 누락 여부 확인 필요

3. `IQR outlier_ratio_%`가 높은 센서 컬럼  
   - clip / winsorize / robust scaler 고려

4. `train_valid_distribution_compare` 차이가 큰 컬럼  
   - split 편향 가능성 확인

5. `sample length distribution`  
   - 너무 짧은 sample 제거 기준 설정 가능

6. `time_diff_sec`, `seq_diff` 이상  
   - 시계열 모델 입력 전 정렬 / 재구성 필요
